# P3: GRPO Training
**ATRD — Adaptive Test-Time Reasoning Distillation**

Phase 3: Group Relative Policy Optimization (GRPO) training loop with PRM rewards

- Model: `nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16` + SFT adapter
- Method: Reinforcement learning via GRPO with composite reward (correctness + format + PRM + redundancy)
- Deliverable: GRPO-trained LoRA adapter exported and synced

> **Prerequisites:** Specs `11-implicit-prm-setup.md` and `12-grpo-training-loop.md` must be read. Phase 2 (SFT) must be complete.

In [ ]:
# Cell 1: Imports and Reproducibility Setup
import random
import numpy as np
import torch
import os, sys, json, re
from pathlib import Path
from typing import List, Dict, Optional, Callable
from dataclasses import dataclass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")

In [ ]:
# Cell 2: Configuration
@dataclass(frozen=True)
class Phase3Config:
    BASE_MODEL: str = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-Base-BF16"
    SFT_CHECKPOINT: str = "/kaggle/working/checkpoints/sft/final_adapter"
    GRPO_CONFIG: str = "configs/base_grpo.json"
    GROUP_SIZE: int = 8
    KL_PENALTY: float = 0.001
    LEARNING_RATE: float = 5e-6
    MAX_STEPS: int = 500
    OUTPUT_DIR: Path = Path("/kaggle/working/checkpoints/grpo")
    LOG_DIR: Path = Path("/kaggle/working/logs")

config = Phase3Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
config.LOG_DIR.mkdir(parents=True, exist_ok=True)
print(f"Phase 3 Config:")
print(f"  Base model: {config.BASE_MODEL}")
print(f"  SFT checkpoint: {config.SFT_CHECKPOINT}")
print(f"  Learning rate: {config.LEARNING_RATE}")
print(f"  Max steps: {config.MAX_STEPS}")
print(f"  Group size G={config.GROUP_SIZE}, KL penalty={config.KL_PENALTY}")

In [ ]:
# Cell 3: Load SFT Model + LoRA
sys.path.append('.')  # Add workspace root to sys.path
from src.models.loader import ModelLoader, setup_blackwell_optimizations
from peft import PeftModel

setup_blackwell_optimizations()
loader = ModelLoader("configs/competition_params.json")
tokenizer = loader.load_tokenizer()

base_model = loader.load_model(quantize=True)

sft_path = Path(config.SFT_CHECKPOINT)
if not sft_path.exists():
    raise FileNotFoundError(
        f"SFT checkpoint not found at {sft_path}. "
        "Run Phase 2 (02_sft_training.ipynb) first to generate the SFT adapter."
    )

model = PeftModel.from_pretrained(base_model, config.SFT_CHECKPOINT, is_trainable=True)
model.print_trainable_parameters()
loader.enable_gradient_checkpointing(model)
print("SFT model loaded and prepared for GRPO training.")

In [ ]:
# Cell 4: Setup PRM Scorer
from src.training.grpo_trainer import GRPOTrainerWrapper

trainer = GRPOTrainerWrapper(
    model=model,
    tokenizer=tokenizer,
    output_dir=str(config.OUTPUT_DIR),
)

reward_fn = trainer.create_reward_function(tolerance=0.01)
print("Reward function created with format + correctness + redundancy components")

In [ ]:
# Cell 5: Load GRPO Training Data
from datasets import load_dataset

dataset_file = Path("/kaggle/working/final_train_dataset.jsonl")
if not dataset_file.exists():
    raise FileNotFoundError(
        f"Training dataset not found at {dataset_file}. "
        "Run P1 notebook first to generate final_train_dataset.jsonl"
    )

dataset = load_dataset("json", data_files=str(dataset_file))["train"]
grpo_train = dataset.select(range(min(2000, len(dataset))))
print(f"GRPO training set: {len(grpo_train)} problems")

# Validate required fields
required_fields = {"question", "answer"}
actual_fields = set(grpo_train.column_names)
missing = required_fields - actual_fields
if missing:
    raise ValueError(f"Dataset missing required fields: {missing}")
print(f"Dataset fields: {grpo_train.column_names}")

In [ ]:
# Cell 6: Train GRPO
print("Starting GRPO training...")
print(f"Group size G={config.GROUP_SIZE}, KL penalty={config.KL_PENALTY}")
print(f"Max steps: {config.MAX_STEPS}")

# Remap dataset fields to match GRPOTrainer expectations
grpo_dataset = grpo_train.map(
    lambda x: {"prompt": x["question"], "ground_truth": x["answer"]}
)

result = trainer.train(
    train_dataset=grpo_dataset,
    reward_function=reward_fn,
)

trainer.save_adapter("checkpoints/grpo/final_adapter")
print("GRPO training and adapter export complete.")

In [ ]:
# Cell 7: Reward & KL Monitoring
import matplotlib.pyplot as plt

# Load trainer state from checkpoint
trainer_state_path = config.OUTPUT_DIR / "trainer_state.json"
if not trainer_state_path.exists():
    raise FileNotFoundError(
        f"trainer_state.json not found at {trainer_state_path}. "
        "Cell 6 must complete GRPO training first."
    )

with open(trainer_state_path) as f:
    trainer_state = json.load(f)

log_history = trainer_state.get("log_history", [])

# Extract reward and KL trajectories
steps = [entry["step"] for entry in log_history if "reward" in entry]
rewards = [entry["reward"] for entry in log_history if "reward" in entry]
kl_steps = [entry["step"] for entry in log_history if "kl" in entry]
kl_values = [entry["kl"] for entry in log_history if "kl" in entry]

# Plot reward trajectory
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(steps, rewards, color="#76B900", linewidth=2)
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Mean Reward")
axes[0].set_title("GRPO Reward Trajectory")
axes[0].grid(True, alpha=0.3)

# Plot KL divergence trajectory
if kl_values:
    axes[1].plot(kl_steps, kl_values, color="#FF4D6D", linewidth=2)
    axes[1].axhline(y=0.05, color="#FFB800", linestyle="--", label="KL threshold (0.05)")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("KL Divergence")
    axes[1].set_title("KL Divergence Trajectory")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "No KL data in log_history", ha="center", va="center")
    axes[1].set_title("KL Divergence (No Data)")

plt.tight_layout()
plt.savefig(str(config.LOG_DIR / "grpo_reward_kl_curves.png"), dpi=150)
plt.show()

# Verify monotonic reward increase
from src.training.grpo_trainer import verify_monotonic_reward
is_monotonic = verify_monotonic_reward(rewards, window=10)
print(f"Monotonic reward increase: {'✓ PASSED' if is_monotonic else '✗ FAILED'}")

# Verify KL < 0.05
if kl_values:
    max_kl = max(kl_values)
    kl_ok = max_kl < 0.05
    print(f"KL divergence max={max_kl:.4f}: {'✓ PASSED (<0.05)' if kl_ok else '✗ FAILED (≥0.05)'}")
else:
    print("KL divergence: No explicit KL logged (using TRL internal KL penalty)")

# Save reward curves to logs
reward_log = {
    "steps": steps,
    "rewards": rewards,
    "kl_steps": kl_steps,
    "kl_values": kl_values,
    "monotonic_reward": is_monotonic,
    "max_kl": max(kl_values) if kl_values else None,
}
with open(config.LOG_DIR / "grpo_rewards.json", "w") as f:
    json.dump(reward_log, f, indent=2)
print(f"Reward log saved to {config.LOG_DIR / 'grpo_rewards.json'}")

In [ ]:
# Cell 8: Evaluation — Baseline vs SFT vs GRPO
from src.training.prm import check_answer, compute_prm_guided_reward

# Load a public benchmark subset for evaluation
eval_file = Path("/kaggle/working/eval_subset.jsonl")
if not eval_file.exists():
    # Use a subset from the training data for comparison
    eval_data = dataset.select(range(min(50, len(dataset))))
    print(f"Using {len(eval_data)} samples from training data for evaluation comparison")
else:
    eval_data = load_dataset("json", data_files=str(eval_file))["train"]
    print(f"Loaded evaluation subset: {len(eval_data)} samples")

# Run inference on evaluation subset with GRPO model
grpo_correct = 0
grpo_total = 0
sample_traces = []
reward_hacking_flags = []

model.eval()
for i, example in enumerate(eval_data):
    question = example["question"]
    ground_truth = example["answer"]

    # Generate completion
    inputs = tokenizer(question, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=4096,
            temperature=0.0,
            do_sample=False,
        )
    completion = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # Check correctness
    is_correct = check_answer(completion, ground_truth)
    if is_correct:
        grpo_correct += 1
    grpo_total += 1

    # Collect first 10 sample traces for manual inspection
    if i < 10:
        sample_traces.append({
            "question": question,
            "ground_truth": ground_truth,
            "completion": completion[:500],  # Truncate for display
            "correct": is_correct,
            "prm_score": compute_prm_guided_reward(completion, ground_truth),
        })

    # Check for reward hacking: correct answer but garbage reasoning
    if is_correct:
        thinking_lines = [l for l in completion.split("\n") if l.strip()]
        if len(thinking_lines) < 3:
            reward_hacking_flags.append({
                "index": i,
                "question": question[:100],
                "reason": "Correct answer with minimal reasoning (< 3 lines)",
            })

grpo_accuracy = grpo_correct / max(grpo_total, 1)
print(f"\nGRPO Evaluation Results:")
print(f"  Accuracy: {grpo_correct}/{grpo_total} = {grpo_accuracy:.1%}")

# Load baseline and SFT results for comparison (if available)
baseline_acc = None
sft_acc = None

baseline_log = config.LOG_DIR / "baseline_results.json"
if baseline_log.exists():
    with open(baseline_log) as f:
        baseline_acc = json.load(f).get("accuracy", None)
    print(f"  Baseline accuracy: {baseline_acc:.1%}" if baseline_acc else "  Baseline: no accuracy field")

sft_log = config.LOG_DIR / "sft_results.json"
if sft_log.exists():
    with open(sft_log) as f:
        sft_acc = json.load(f).get("accuracy", None)
    print(f"  SFT accuracy: {sft_acc:.1%}" if sft_acc else "  SFT: no accuracy field")

# Verify GRPO > SFT
if sft_acc is not None:
    improvement = grpo_accuracy > sft_acc
    print(f"  GRPO > SFT: {'✓ PASSED' if improvement else '✗ NEEDS INVESTIGATION'}")

# Reward hacking check
print(f"\nReward hacking flags: {len(reward_hacking_flags)}")
for flag in reward_hacking_flags[:5]:
    print(f"  - Sample #{flag['index']}: {flag['reason']}")

# Print sample traces for manual inspection
print(f"\n{'='*60}")
print("SAMPLE REASONING TRACES (first 10):")
print(f"{'='*60}")
for i, trace in enumerate(sample_traces):
    print(f"\n--- Sample {i+1} ({'✓' if trace['correct'] else '✗'}) PRM={trace['prm_score']:.3f} ---")
    print(f"Q: {trace['question'][:150]}")
    print(f"Expected: {trace['ground_truth']}")
    print(f"Output: {trace['completion'][:300]}")

# Save evaluation results
eval_results = {
    "grpo_accuracy": grpo_accuracy,
    "grpo_correct": grpo_correct,
    "grpo_total": grpo_total,
    "baseline_accuracy": baseline_acc,
    "sft_accuracy": sft_acc,
    "reward_hacking_count": len(reward_hacking_flags),
    "reward_hacking_flags": reward_hacking_flags,
    "sample_traces": sample_traces,
    "reward_curves": {
        "steps": steps if 'steps' in dir() else [],
        "rewards": rewards if 'rewards' in dir() else [],
    },
}
eval_output_path = config.LOG_DIR / "p3_grpo_eval.json"
with open(eval_output_path, "w") as f:
    json.dump(eval_results, f, indent=2, default=str)
print(f"\nEvaluation results saved to {eval_output_path}")

In [ ]:
# Cell 9: Export to Hugging Face Hub
from scripts.sync_to_hub import sync_adapter

api_token = os.environ.get("HF_TOKEN")
if not api_token:
    raise EnvironmentError(
        "HF_TOKEN not set. Set your Hugging Face API token via:\n"
        "  os.environ['HF_TOKEN'] = 'hf_...'\n"
        "or add it to Kaggle secrets."
    )

adapter_path = Path("checkpoints/grpo/final_adapter")
if not adapter_path.exists():
    raise FileNotFoundError(
        f"GRPO adapter not found at {adapter_path}. "
        "Cell 6 must complete training and save the adapter first."
    )

print("Syncing GRPO adapter to Hugging Face Hub...")
sync_adapter(
    adapter_path=str(adapter_path),
    repo_id="samar/atrd-nemotron-grpo-r32",
    commit_message="GRPO Phase 3: RL-optimized policy after 500 steps with G=8",
    private=True,
)
print("Adapter synced to Hugging Face Hub.")

In [ ]:
# Cell 10: Cleanup
import gc

# Verify all required outputs exist
required_outputs = [
    Path("checkpoints/grpo/final_adapter/adapter_config.json"),
    config.LOG_DIR / "p3_grpo_eval.json",
    config.LOG_DIR / "grpo_rewards.json",
]
for output in required_outputs:
    status = "✓" if output.exists() else "✗"
    print(f"  {status} {output}")

# Release GPU memory
del model, base_model, trainer
torch.cuda.empty_cache()
gc.collect()
print("\nGPU memory cleared.")
print("P3 Complete — Phase gate: python scripts/verify_unit_completion.py P3 grpo")